# COMPUTER VISION · LAB 02
## Image quality: detail, noise, and sampling

**New Uzbekistan University** · Mohammad Sabokrou  
**Level:** easy to medium · **Time:** about 2 hours · **Environment:** Google Colab / Python 3

An inspection camera must recognise a part, read its label, and detect a small crack.
How do focus, sensor noise, sampling, and brightness precision affect those tasks?
Can a smoother image hide a real defect? You will answer these questions with experiments.

**Starting point:** you already know how to open a notebook, display an image, and work with
image arrays from Lab 0. The setup is supplied. This lab moves on to image-quality decisions:
you will not repeat image-loading, datatype-conversion, brightness-arithmetic, file-saving,
or raw-pixel-matching exercises.

**The goal is to explain computer vision behaviour.** You do not need to write loops,
implement algorithms, or memorise Python. Change a few values, inspect the evidence, and explain.

By the end, you should be able to:

- Explain why a high pixel count does not guarantee sharp detail.
- Distinguish scene detail from sensor noise and recognise denoising trade-offs.
- Compare Gaussian smoothing and median filtering on different noise patterns.
- Distinguish resolution, interpolation, aliasing, and quantisation.
- Select a setting for a visual task and test whether your conclusion transfers to another image.


### How to work

1. Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/), then connect to a **CPU** runtime. A GPU is unnecessary.
2. Work from top to bottom. Read the short explanation **before** running each cell.
3. **RUN** cells are ready. **FILL** cells need one or two values. **EXPERIMENT** cells need a small change and a comparison.
4. After **every code cell**, answer the question immediately below it. Double-click a text cell and replace **[Write here]**.
5. Record a prediction before changing a setting. Then keep the output and write what you observed, even if your prediction was wrong.

**Answer length:** usually 1–3 sentences. Name a visible feature or quote a number;
“better” or “worse” alone is not enough. Your own explanation matters more than perfect English.

`None` means “not filled yet”. An unfinished cell prints a reminder, so **Run all does not mean the work is complete**.
Only change lines marked **FILL** or **CHANGE**. You may ask for help with Python syntax.

| Part | Focus | Minutes |
|---|---|---:|
| 0 | Setup and identity | 5 |
| A | Focus and useful detail | 10 |
| B | Sensor noise and denoising trade-offs | 20 |
| C | Isolated faulty pixels and median filtering | 10 |
| D | Resolution and enlargement | 20 |
| E | Sampling and aliasing | 15 |
| F | Quantisation | 10 |
| G | Your controlled investigation | 20 |
| Finish | Conclusions and submission check | 10 |

**Key habit:** change one factor at a time. Keep the scene and display scale fixed so the comparison is fair.


## 0 · Open your visual laboratory

**RUN 0.1.** The tools and images are embedded. Run once, then move to the experiments.
No installation, upload, Drive mounting, or image-loading exercise is required.
You do not need to understand or edit the supplied implementation.

The inspection target is an original designed test image, not a real camera measurement.
The photograph is a real sample image. Both stay unchanged; experiments create copies.


In [ ]:
import base64, io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

PHOTO_DATA = "..."
TARGET_DATA = "iVBORw0KGgoAAAANSUhEUgAAAYAAAAGACAAAAACBrOpjAAAPKklEQVR4nO2dfXAU5R3HvxcIkASkvMSA8iJgwATCewB5Kx21rTo6dUarbUarNVBR2zo6tbUztdrWYuvUWnWUqdiho+kLUGu1rYKUXMJbwkuJaRAxIm8iQgwvxoAhCdc/LnfZu9vdu919dn97ue9nmLB7t8/vufw++3v2dvfuSWAjiCRZ0i8g06EAYShAGAoQhgKEoQBhKEAYChCGAoShAGEoQBgKEIYChKEAYShAGAoQhgKEoQBhKECY3kmeL/PkVfRkKsyfDpjdlGf21WDmwGwIYv4VYZZI4wpg+lViWASGAsqSNCQpkiSRRgLKTFsRK5jm0vRtKPOvBNM0GggoS9qQpE4FYHhM1RfA/KvFxIDxEMT8K8Q4mbwUIYyugDKwABRTAYMxiBUgDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwiT7WEoc9zR1LQRyc4ePm1kY++zhBwAMfi7LsEnB2GkTwytN9xh1UfiYwRN60V96PXG7p4cZhfYjFgVECbW2NtX//ZLy8doHKwHgRP1U4yYNrw3/9hSbXZpH7ya9itrZqz3w8FrNWmf4/nLQtMnRx/5mr6+UogMoyLcXXwgbFfCTEgChzw5srurE+T8MLo0+ses0AGB7a55+k9bDtevPAX+9cAGQvyr65O5HgYEvJO1WN/qtt2o3aV3SDiwKWPllxLFbAYEBJXc9nAOEVnZEH6sEUAi0bzZo0r/o9mUDAbzcbqfHJNEBYEs7EFhkJ7gcToagonIATdsjqy3/BQaXw3SUGHk3gJPbbHSWSvQggMlDbAQXxNExYH4+gPrIWnUnMG/MSOD9D42bTLsYQIONvlKI/lEjgC/ZiC2JIwGBSQCi+QgCWIAFMD9QXqZtYoEUogcB5JUaPetTnL0LGgLg067l/QeBEZdgfgCoPm/cZDCAFus9pRA9tBHA/GzrsUVxJiCkWQ4CWAAMvQw4VWfc5DyATus9pRC9vhnpNwI5FHACwAXhxc5NQGA+kGwMOgVggOWOUokeBDB6rOXQwjgSEGoAMCK8vKMFmJAP4PLewI7PDNu8A2CU5Z5SiH52O4BFliNL40jApiYAk8PLQYR3T+RNAzo2GTXZeRTATMs9pRB98zmg10LLkaVxImDPCgD54WyergN6XQ4AWIiu6zY6HFoOYOR0qz2lEr0KwEzrg5s0ti/GfXZwU1UnELgt/LajuhOY1h8AMD2vFfsPJQ4zoTOHatefA/rea9l6CtGP7kU6jkB2BPxcsxz41uzwQhBdYwSQPXsDELzNqEn+fWMsd2kePbrJoGmWI4vj7F3QqEevCS/sOwzkzOh6dAGAjQZvNXPu+G2h/jMmpBA9VA1gYXpdiQZgfwgK9MsbPm7GhMhqEMDsPl0rxUOacXqX/pH27H8mX2y5sxSiNzQjLUcg25ejY2nfhOgYAQTmvQYEZ8Y3aTte/89mHHr4Z1YNJIsOhB2Nt65WHjVFu6MVGDQpuroAwM6E6w19R177ZDHQ8uvP1Uc/uw1peBYM2B+CYqkEcPLmmIc6N16TuF3ODx5swtGKO5VH39oG9J1rLaw/UFIBJ+t1HtR9s563BMC6g8qjBwHMybEU1icoEVCld33y4AG9TadMBUJ/Uh392LtIz0OwoiGoCsCDMYfFl14HgrfrbXtLHbCr0cpb0RSiBwEUFFuI6R9UVMB7R4C8qTEPzYfRqcDY6QBWq40eqkba3YyPoEJAEMDs2FIacxHQslN365sA1DUqjb67Kf1uxkdQIODcFgDz4h6cB6MrcuOmwUoJpBI9CKAkzW7GR1AgYNsZYODEuAfnA6g7rbv9jbBSAilE/7wWaXoSACUCKgHMjY8zfGz0s2zxFE6BhRJIIfrWNiBvVqoBfYZzAc0NCO+SsRiPQZZKIJXoQaThzfgIzgUEQ0B+4tvKeQHg8Ae6LSaUIOUSSCH68fQ9CYDBhE2cqkA9RjlNwyvoPQsKEIYChKEAYShAGAoQhgKEoQBhKEAYChCGAoShAGEoQBgKEIYChKEAYShAGAoQhgKEoQBhKEAYChCGAoShAGEoQBgKEIYChJETULN4j8MI9Yt3df2I4ae/sNlQBCsC3l9s5btdWnYurjdY8QCv+7OGmi9qpxmTk8/T6xk8BghjuQJqXrz/1L8/GXrdLAAdb2w7kTP6q4WoefH+Y+tOFlxbCgAd62qP95lww3AAHW/VHs8tvL72X3gGmHsHAODV7pWtkUDaNkBc5OPrTo4uG3F49Qf9rrgaCNVXHTpz4aJFeq/txOrdmPjNrgiRiJH+NA3rn7l7Wlw/YtgYgrbUAB+v+MJ4YFUlcK7+necBVO0Ejvy+fS7Q+dReoGPXuz8eho4nG4HTO5oNvkLdHUjTJowmcvUO4P2n7nviLNpeuXAGDj4L4EhFy3WJ8c786gSw41hnNnQi6jfU9COEDQHbbyntXflq1XigbtCSUecOrAeAXTdeHqhdvao0Gxv2llx/UVvNmlfuxvrGYTeNDb3X8LWRy787OdK8e6U7kKZNGE3kulunn//L9icm3jCw/oUNM5BVuuiiXvtffvOqfgkv7I0TRTcPPfjSsdHQRoz0p9tQ048QNgRcdQVw7eYjAPpfcCn6TJoEAPO+AlzZtKGxGDUD781Cn6sO7zifVZv9/aHAjBlJA2nahJ/SRP7yQuDG7Tl39kZp9UfAqCUAim96bn9RQry3c+/Kxfi7HoFeRP2Gmn6EsCGgEACGNAH4xvJHi0cXDQCAYgAo2tAEHG3/Tni71gHHRgxNLZCmTfh/TeRxAAZhdG8Ag/YC2LLxo7Oh8ASwcTSNzwVw8UDoRdRvqOlHCBsCsgEgAACFj7+7b8vKhTfrz9LQofuobqCENprI2QACXduGgLVrkkUPxaxFt9NtaP4beIGz84DskhLUP1M4A3hnJoA9yAcK2h6L/DoFHzaHJxAIQDPlTMxK14bdbRIix7HxgsWj+mW9/azOq8k/cCYXOPLpoNiIXf3pNzTsxyucnAd0PF7b1HGqAZ8A2Ly2pWV9MK8QmNP03L6zbUfeXAnMbn+6obVl5x+Riz3nos1iVsJo2iRETnjFfft8/vaf9V7PlDPLj7Q1Lo+PGOlPp6FZPx7hpAJC+/YBQPYUAFPXrAHw9Wzgyj11dQBQBFxZt+93AMZgdPaGDZHzgNiVMJo2CZHjmP7GLwHMatZ5PVdv2/MIMKIgLmJXf3oNTfrxCicCsn9U3XhywKVXDwOwaMJbpwqumQWg1/cqtx7tNbTki0DvB9Zua+pfeB1ylvzj4+jYG7MSRtMmIXIc12fVnM6/YqTeX0HJ/eGq3YGist/ERezqT6+hST9eoeab8jUv3p/4rpBo4TflfQoFCEMBwqi5HzBnjpIwmQgrQBgKEIYChKEAYShAGAoQhgKEoQBhKEAYChCGAoShAGEoQBgKEIYChKEAYShAGAoQhgKEoQBhKEAYChCGAoShAGEoQBgKEIYChKEAYShAGJdmS3kodnWZmqi1asK4zGxLW6sX8JDRY4os9DAUC9DLvuY5OkhApQCz7Ee3oINY1AlInv6uzahAiyoBKaY/vCkVdKNGgIX0hzengggqBFhMf7gJFYRRcCJmI/+2W/U8HFeA7USyCAA4rwAnOzKLAI4FOMshDTgcghwnkMOQowpQsQNnfBE4EKAmd3438LzLk7raF6Aqc3434DK2BajLm68NPI+l7nZgV4DKrPnagNvwlqQprheAXQFqd9pMLgF7AlRnzK8G3C8AewLU58uvBtyHxwATPCgAWwLc2F0ztgRsCHAnVz404EUBcAiSxroAt3ZV35WAJwXACpDGsgD3dlSflYA3BcAKkMaqADd3U1+VgEcF0HMqoLxcb9H/WBTg7k7qIHo5UJ64aB+vCqCnVEB59Id2MR3oIQJU41kBWBTg9mHSdvwV0R/axXSgp1TAiu6kr1CSf6/+wm2P+ZPmK3QX7bLUs78wbKkC3H+j7qNTAY8U9JQhSDVeHYMpwBhvSoACDFjqkQEKMMKHZ8JeHCF9dBT2pgRYAYbwaqg8HpQABRjjyXGYAkzgTXl5XC8BCjCDH8ySx+0SoABT3D8OU4A5/vx+QGbhbglQQBJ8+h2xjMLVEqCAZPj0e8KZg8ufULEiwIuZTTJu9hRWQBLc/ogWBQhDAcJQgDn+mi3F/SNkxh2DWQHm+Ha2FKIKawLcHiEybwRiBZji1+lqiDosCnB3jPDbCOTX6WqIQqwKcHMn9VsBeAIrwBh/zhfk3m6akQXACjDGr/MFubWjZmYBsAKksSHAnV3VdwXg4wmb3MiV7/LvFRyCDPD1hE3qd9eMLQCbFaA6Xz7M/1Iffk1Vg9qM+TD/nsFjgDB2BajcaTO5AOxXgLqsZXT+HQxBqvKW2fl3cgxQk7kMz7+jg7CK3GV6/p3NGbfM6eQmGZ9+x29DnWWQ+Xd+HuAkh8w/FExbaXsYYvoBKDkTtpdJ5j+MiolbbRQB0x9Bzcy5FhUw/d2omrrYggKmX4u6uaNTVMD0x6Jy8u5lSWedZPYTUDx7upkDZl8P9dPXLwMSLDD3hrj09wPcyfhsV6LKwluSwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwlj6efjuwUuKnMYsDWYFAVvhH9F/SRYVP6GzzPyspZQVIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcJQgDCBjToPlgGo8PqV9HCMcsoKEIYChKEAYShAGAoQhgKEoQBhKEAYChCGAoShAGEoQBgKEIYChNEVUIHw5VOiDMMr/KwAYYwFsAQUYpxMfQEV5o2IRcoAo3uMBhVAAyoxyb/5MYAGlGCaRt2b8ppWvDfvlCSJNBTA3V8phjuysQAqUIfJOGJ2DODwowizRJpVAMAqcE6S3TiZAOIyvBQhDAUIQwHCUIAwFCAMBQhDAcJQgDAUIAwFCEMBwlCAMBQgDAUIQwHCUIAwFCAMBQhDAcL8H/ymKpfdQJuMAAAAAElFTkSuQmCC"
photo = np.asarray(Image.open(io.BytesIO(base64.b64decode(PHOTO_DATA))).convert("RGB"))
target = np.asarray(Image.open(io.BytesIO(base64.b64decode(TARGET_DATA))).convert("L"))
gradient = np.tile(np.linspace(0, 255, 384, dtype=np.float32), (90, 1)).astype(np.uint8)
completion = {}

def ready(key, **values):
    """A reminder only: this does not grade your scientific answers."""
    missing = [name for name, value in values.items() if value is None or value == ""]
    completion[key] = not missing
    if missing:
        print("FILL " + ", ".join(missing) + " and run this cell again.")
    return not missing

def pil(a):
    return Image.fromarray(np.uint8(np.clip(np.rint(a), 0, 255)))

def panels(images, titles, hist=False):
    n = len(images)
    fig, axes = plt.subplots(2 if hist else 1, n, squeeze=False,
                             figsize=(3.35*n, 5.6 if hist else 3.7),
                             layout="constrained")
    for j, (a, title) in enumerate(zip(images, titles)):
        axes[0, j].imshow(a, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axes[0, j].set_title(title, fontsize=10)
        axes[0, j].axis("off")
        if hist:
            axes[1, j].hist(np.asarray(a).ravel(), bins=np.arange(-0.5, 256.5, 4), color="#246f85")
            axes[1, j].set(xlim=(0, 255), xlabel="Stored value (0 to 255)", ylabel="Pixel count")
    if hist:
        ymax = max(ax.get_ylim()[1] for ax in axes[1])
        for ax in axes[1]: ax.set_ylim(0, ymax)
    plt.show()

def blur(a, radius):
    return np.array(pil(a).filter(ImageFilter.GaussianBlur(radius=float(radius))))


def reduce_image(a, width):
    height = max(1, round(a.shape[0]*width/a.shape[1]))
    return np.array(pil(a).resize((int(width), height), Image.Resampling.BOX))

def enlarge(a, size, method="nearest"):
    choices = {"nearest": Image.Resampling.NEAREST, "bilinear": Image.Resampling.BILINEAR}
    return np.array(pil(a).resize(size, choices[method]))


def quantize(a, bits):
    levels = 2**int(bits)
    return np.rint(np.rint(a.astype(float)/255*(levels-1))*255/(levels-1)).astype(np.uint8)

def block_mean(a, step):
    h, w = a.shape
    return a.reshape(h//step, step, w//step, step).mean(axis=(1, 3))

def stripe_experiment(step):
    x = np.arange(320)
    stripes = np.tile(127.5+110*np.cos(2*np.pi*x/10), (160, 1))
    sampled = stripes[::step, ::step]
    averaged = block_mean(stripes, step)
    panels([stripes, sampled, averaged],
           ["Reference: 32 stripe cycles", f"Pick every {step}th pixel", f"Average {step}×{step}, then reduce"])
    print(f"Step {step}: reduced shape = {sampled.shape}; same scene extent in every panel.")

def trial_image(source, factor, setting):
    if factor == "resolution":
        a = reduce_image(source, int(setting))
        return enlarge(a, (source.shape[1], source.shape[0]))
    if factor == "blur":
        return blur(source, float(setting))
    if factor == "bits":
        return quantize(source, int(setting))
    raise ValueError("Use resolution, blur, or bits.")


def noisy_image(a, strength):
    # Same noise pattern for every strength: only the noise amplitude changes.
    pattern = np.random.default_rng(17).normal(size=a.shape)
    return np.clip(np.rint(a.astype(float)+strength*pattern), 0, 255).astype(np.uint8)

def impulse_image(a, fraction=0.04):
    r = np.random.default_rng(29).random(a.shape)
    out = a.copy()
    out[r < fraction/2] = 0
    out[(r >= fraction/2) & (r < fraction)] = 255
    return out

def flat_spread(a):
    # Known uniform region of the clean inspection target; no object edges in this patch.
    return float(np.std(a[124:140, 50:334]))

def detail_panels(images, titles):
    from matplotlib.patches import Rectangle
    fig, axes = plt.subplots(2, len(images), squeeze=False,
                             figsize=(3.35*len(images), 5.5), layout="constrained")
    for j, (a, title) in enumerate(zip(images, titles)):
        axes[0,j].imshow(a, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axes[0,j].add_patch(Rectangle((170,150), 172, 102, fill=False, edgecolor="#e08300", linewidth=1))
        axes[0,j].set_title(title, fontsize=10)
        axes[0,j].axis("off")
        axes[1,j].imshow(a[150:252,170:342], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axes[1,j].set_title("Same detail window · enlarged", fontsize=9)
        axes[1,j].axis("off")
    plt.show()

panels([photo, target], ["Photograph", "Inspection target"])
print("Ready. Photograph shape:", photo.shape, " | Target shape:", target.shape)


### Q0.1 · WRITE

Find two features on the target that you think will disappear before the large dark circle
when image quality decreases. This is your first prediction.

**My two features and reason:** [Write here]


**FILL 0.2 — Your details.** Put your name and student ID between the quotation marks.
Your ID stays as text, so leading zeros are preserved. This records your details;
it does not check registration against the university system.


In [ ]:
student_name = ""  # FILL
student_id = ""    # FILL — keep quotation marks
if ready("identity", student_name=student_name.strip(), student_id=student_id.strip()):
    print("Student:", student_name, "| ID:", student_id)
    print("Suggested filename:", student_id + "_Lab2_CV.ipynb")


### Q0.2 · WRITE

Check the displayed identity. Why is detecting a tiny crack likely to require more image detail
than deciding whether a large circle is present?

**My explanation:** [Write here]


## A · Focus and useful detail

The lens spreads light from one scene point over a small area when focus is poor.
Nearby features may merge even when the sensor still records many pixels.

### A1 · RUN — Same dimensions, different sharpness

We apply Gaussian blur to a saved image to illustrate lost sharpness. This resembles
one effect of poor focus but is not a full optical model of a lens.
The blur radius is in pixels, not a lens aperture or focal length.
Inspect the small label, bright markers, crack, and fine stripes.


In [ ]:
panels([target, blur(target, 1), blur(target, 3)],
       ["Reference · radius 0", "Mild blur · radius 1", "Strong blur · radius 3"])
print("Every image has the same dimensions:", target.shape)


### QA1 · WRITE

Name one feature that becomes harder to distinguish and one that remains visible.
A student says, “The number of pixels stayed the same, so the useful detail stayed the same.”
Use the output to respond.

**Feature comparison:** [Write here]  
**My explanation:** [Write here]


### A2 · FILL — How much blur can your task tolerate?

Your task is to separate the **two small bright markers** in the grey rectangle.
Choose a blur radius from **0.5, 1.5, 2.5, 4.0**. Predict whether you will still see two spots.
Only fill the value; the tool shows the same close-up for both images.
The orange box marks the detail window. This viewing aid makes small features easier to compare.


In [ ]:
focus_radius = None  # FILL: 0.5, 1.5, 2.5, or 4.0
if ready("A2", focus_radius=focus_radius):
    if focus_radius in [0.5, 1.5, 2.5, 4.0]:
        detail_panels([target, blur(target, focus_radius)],
                      ["Reference", f"Blur radius {focus_radius}"])
    else:
        completion["A2"] = False
        print("Choose 0.5, 1.5, 2.5, or 4.0.")


### QA2 · WRITE

**Radius and prediction before running:** [Write here]  
**Observation:** Can you distinguish two separate bright markers? [Write here]  
**Conclusion:** Would this level of sharpness be adequate for detecting the large circle
but inadequate for the marker task? Explain from the output. [Write here]


## B · Sensor noise: a smooth image is not always a useful image

**Noise** is unwanted variation in the recorded values. It can hide faint features or create
spots that resemble defects. It is different from a real crack, which belongs to the scene.

Here we add a repeatable random pattern to the saved target. “Noise strength” controls the
typical size of the added variation in 0–255 value units. This is a simple additive Gaussian
noise model, not a full model of photon noise or any particular camera.
The supplied code keeps the random pattern fixed so only its strength changes.

### B1 · RUN — Compare noise levels

The printed **flat-patch spread** measures variation in a region known to be uniform in the
clean target. Larger values mean more variation in that patch. It is not a measure of crack visibility.
No statistical formula needs to be programmed.


In [ ]:
noise_strengths = [0, 12, 28]
noise_examples = [noisy_image(target, strength) for strength in noise_strengths]
detail_panels(noise_examples, [f"Noise strength {s}" for s in noise_strengths])
for strength, a in zip(noise_strengths, noise_examples):
    print(f"Strength {strength}: flat-patch spread = {flat_spread(a):.2f}")


### QB1 · WRITE

1. Compare the strongest-noise image with the reference: did the real crack move, or did
   unwanted variations make it harder to see?
2. Quote the flat-patch spread at strengths 0 and 28. Why is a known uniform patch useful
   for checking noise, whereas the striped region is a poor choice?

**My observation and two measurements:** [Write here]  
**Why the patch choice matters:** [Write here]


### B2 · RUN — Reduce noise, but watch the real features

Gaussian smoothing combines nearby values, giving more weight to nearby pixels.
It can reduce random variation but also mixes real edges and small features.
We keep the noisy input fixed and compare two smoothing radii.

Before running, predict whether the smoothest image will always be best for detecting the thin crack.


In [ ]:
fixed_noisy = noisy_image(target, 28)
denoised_mild = blur(fixed_noisy, 0.8)
denoised_strong = blur(fixed_noisy, 2.5)
detail_panels([fixed_noisy, denoised_mild, denoised_strong],
              ["No smoothing", "Smoothing radius 0.8", "Smoothing radius 2.5"])
for label, a in [("None", fixed_noisy), ("Mild", denoised_mild), ("Strong", denoised_strong)]:
    print(f"{label}: flat-patch spread = {flat_spread(a):.2f}")


### QB2 · WRITE

**Prediction before running:** [Write here]

Which output has the lowest flat-patch spread? Compare that output with the mildly smoothed
image: which better preserves the crack and the two markers?
Explain why a lower noise number alone cannot tell you whether the image is good for inspection.

**Measurements and feature evidence:** [Write here]  
**My conclusion about the trade-off:** [Write here]


### B3 · EXPERIMENT — Choose your own denoising strength

Keep noise strength fixed at **28**. Choose two different smoothing radii from **0.5, 1.0, 2.0, 3.0**.
Your task is to reduce the noise while keeping the **thin crack visible**.
The reference below is the noisy input; compare it with your two processed outputs.
Do not use only the flat-patch number to make your decision.


In [ ]:
mild_radius = None    # FILL: 0.5, 1.0, 2.0, or 3.0
strong_radius = None  # FILL: a LARGER value from the same list
if ready("B3", mild_radius=mild_radius, strong_radius=strong_radius):
    allowed_radii = [0.5, 1.0, 2.0, 3.0]
    if mild_radius in allowed_radii and strong_radius in allowed_radii and mild_radius < strong_radius:
        test_input = noisy_image(target, 28)
        test_outputs = [test_input, blur(test_input, mild_radius), blur(test_input, strong_radius)]
        test_labels = ["No smoothing", f"Radius {mild_radius}", f"Radius {strong_radius}"]
        detail_panels(test_outputs, test_labels)
        for label, a in zip(test_labels, test_outputs):
            print(f"{label}: flat-patch spread = {flat_spread(a):.2f}")
    else:
        completion["B3"] = False
        print("Choose two allowed radii, with mild_radius < strong_radius.")


### QB3 · WRITE

**Prediction before running:** [Write here]

| Smoothing radius | Flat-patch spread | Crack visibility: clear, uncertain, or lost? |
|---|---:|---|
| 0 (no smoothing) | ... | ... |
| My smaller radius: ... | ... | ... |
| My larger radius: ... | ... | ... |

**Conclusion:** Choose one setting and justify it with both the noise measurement and
the visible crack. What stayed fixed to make your comparison fair? [Write here]


## C · Different noise may need a different filter

Some corrupted images contain isolated very dark or bright pixels, often called
**salt-and-pepper noise**. This can illustrate impulse corruption or faulty measurements;
it is not the same pattern as the Gaussian noise in Part B.

A **median filter** replaces each pixel with the middle value among nearby pixels.
An isolated extreme value often disappears because it is not the middle value.
A window of 3 means a **3 × 3 neighbourhood**. A larger window can also remove real small details.

### C1 · RUN, then CHANGE — Gaussian or median?

About 4% of pixel locations are selected for impulse corruption. Compare Gaussian smoothing
with median filtering on **the same corrupted input**. Run with `median_window = 3`,
then with `median_window = 5`, recording each result. The Gaussian radius stays at 1.0;
its radius and the median window size are different kinds of parameters, not equivalent strengths.


In [ ]:
median_window = 3  # CHANGE: run with 3, then 5
if median_window in [3, 5]:
    impulse_noisy = impulse_image(target, fraction=0.04)
    gaussian_result = blur(impulse_noisy, 1.0)
    median_result = np.array(pil(impulse_noisy).filter(ImageFilter.MedianFilter(size=median_window)))
    detail_panels([impulse_noisy, gaussian_result, median_result],
                  ["Impulse-corrupted input", "Gaussian radius 1.0", f"Median {median_window}×{median_window}"])
    if "median_windows_tested" not in globals():
        median_windows_tested = set()
    median_windows_tested.add(median_window)
    completion["C1_two_windows"] = {3, 5}.issubset(median_windows_tested)
    print("Median windows tested:", sorted(median_windows_tested))
else:
    print("Choose 3 or 5.")


### QC1 · WRITE

| Method | What happened to isolated bright/dark dots? | What happened to the crack and markers? |
|---|---|---|
| Gaussian radius 1.0 | ... | ... |
| Median 3 × 3 | ... | ... |
| Median 5 × 5 | ... | ... |

**Conclusion:** Which tested filter would you choose for these isolated dots, and why?
Did the larger median window remove only unwanted dots, or also change genuine detail?
Avoid claiming that one filter is best for every noise type. [Write here]


## D · Spatial resolution: how many samples describe the scene?

**Spatial resolution here means the image's pixel dimensions.** Reducing dimensions gives
fewer samples across the same scene. A small feature may merge with its surroundings.
The helper averages small regions when reducing an image.

### D1 · RUN — Same scene, different pixel grids

Every panel occupies a similar display area. Coarse pixels are enlarged using nearest
neighbour for viewing. This does not restore the original measurements.


In [ ]:
widths = [384, 96, 48, 24]
small_targets = [reduce_image(target, w) for w in widths]
panels(small_targets, [f"{a.shape[1]} × {a.shape[0]} pixels" for a in small_targets])
for a in small_targets:
    print(f"{a.shape[1]}×{a.shape[0]}: {a.size:,} grayscale samples")


### QD1 · WRITE

Fill the table using your display. “Unsure” is an acceptable observation.

| Width | Large circle visible? | PART A7 readable? | Two bright markers separable? |
|---|---|---|---|
| 384 | ... | ... | ... |
| 96 | ... | ... | ... |
| 48 | ... | ... | ... |
| 24 | ... | ... | ... |

**Reasoning:** Why does the smallest adequate resolution depend on whether you need to
recognise the circle, read the label, or separate the markers? Use two rows of your table. [Write here]


### D2 · FILL — Choose a capture size for a task

Your task is to read **PART A7** while reducing the number of samples.
Choose a width from **24, 48, 96, 192**. Start with the smallest one you predict will work.
You may revise your choice after looking. This is a visual judgement, not an automatic OCR test.


In [ ]:
chosen_width = None  # FILL: 24, 48, 96, or 192
if ready("D2", chosen_width=chosen_width):
    if chosen_width in [24, 48, 96, 192]:
        chosen = reduce_image(target, chosen_width)
        panels([target, chosen], ["Reference 384 × 384", f"Your choice {chosen_width} × {chosen_width}"])
        print(f"Samples retained: {100*chosen.size/target.size:.2f}% of the reference")
    else:
        completion["D2"] = False
        print("Choose 24, 48, 96, or 192.")


### QD2 · WRITE

**Initial prediction and chosen width:** [Write here]  
**Final width, retained percentage, and evidence that the label is / is not readable:** [Write here]  
**Would you keep the same width for separating the two bright markers? Explain:** [Write here]


### D3 · RUN — Does enlargement create information?

**Nearest neighbour** repeats a nearby value. **Bilinear interpolation** blends nearby
values, producing smoother transitions. Both estimate a larger display from the same small image.
Neither takes a new high-resolution measurement of the original scene.


In [ ]:
low = reduce_image(target, 24)
near = enlarge(low, (384, 384), "nearest")
smooth = enlarge(low, (384, 384), "bilinear")
panels([target, near, smooth],
       ["Original measurement grid", "24 → 384 · nearest", "24 → 384 · bilinear"])
print("Both enlarged outputs:", near.shape, "| Both started from", low.size, "samples.")


### QD3 · WRITE

Which enlargement looks smoother? Does that prove it restored the small label or crack?
Explain the difference between **more output pixels** and **more captured detail**.

**My explanation, using one target feature:** [Write here]


## E · Sampling can invent misleading patterns

**Aliasing** happens when sampling is too coarse for the detail present: a fine pattern
can appear as a different, coarser pattern or disappear. It is not simply “an image looks blurry”.
This matters for striped fabric, fences, repetitive industrial textures, and small structures.

The reference below has **32 bright–dark cycles** across its width.
“Pick every 8th pixel” keeps only one sample from each group of eight columns.
“Average 8×8” first combines local values. Averaging is a simple anti-aliasing filter;
it reduces this artifact but also removes fine detail and is not a perfect filter for every pattern.

### E1 · RUN — Direct sampling versus area averaging


In [ ]:
stripe_experiment(step=8)


### QE1 · WRITE

1. Does direct sampling preserve the 32 original stripe cycles, or show a different coarse pattern?
2. Compare the strength of the stripes after area averaging. What is gained, and what is lost?

**My observation and trade-off:** [Write here]


### E2 · EXPERIMENT — Change the sampling step

Run the next cell with `step = 2`, then `step = 4`, then `step = 16`.
Record each result before rerunning. A larger step means fewer retained samples.
The reference pattern and display range stay fixed.
The notebook remembers the steps you tested and prints them.


In [ ]:
step = 2  # CHANGE: test 2, 4, and 16, one at a time
if step in [2, 4, 16]:
    stripe_experiment(step)
    if "steps_tested" not in globals():
        steps_tested = set()
    steps_tested.add(step)
    completion["E2_three_steps"] = {2, 4, 16}.issubset(steps_tested)
    print("Steps tested in this runtime:", sorted(steps_tested))
else:
    print("Use step 2, 4, or 16.")


### QE2 · WRITE

**Prediction before changing the step:** [Write here]

| Step | Direct sampling: faithful, changed pattern, or uncertain? | Effect of averaging |
|---|---|---|
| 2 | ... | ... |
| 4 | ... | ... |
| 16 | ... | ... |

**Conclusion:** For which tested step does direct sampling best preserve the original stripes?
Why can a camera show a pattern that is not actually on the object? [Write here]


## F · Quantisation: how many brightness levels are available?

Resolution controls **where** we sample. Quantisation controls **which values** we can store.
With $b$ bits, there are $2^b$ available levels: 8 bits gives 256; 2 bits gives 4.
Here the dimensions stay fixed while brightness values are rounded to fewer levels.
This tests loss of tonal precision; it is not the grayscale conversion or datatype conversion practiced in Lab 0.

### F1 · RUN — A smooth ramp with fewer levels

The images are displayed on the same 0–255 scale. Low-bit values are mapped back to that
scale only for display. The resulting arrays remain `uint8` containers; their actual file
sizes do not automatically become 2-bit or 4-bit files.


In [ ]:
bit_choices = [8, 4, 2]
ramps = [quantize(gradient, b) for b in bit_choices]
panels(ramps, [f"{b} bits · {2**b} available levels" for b in bit_choices])
for b, a in zip(bit_choices, ramps):
    print(f"{b} bits: {len(np.unique(a))} distinct levels used; dimensions {a.shape}")


### QF1 · WRITE

Describe the visible bands. Did they appear because there are fewer pixel positions or
because there are fewer possible brightness values? Would doubling the image width alone
remove these bands if the bit depth stayed at 2 bits?

**My explanation:** [Write here]


### F2 · FILL — Test a different bit depth

Choose **3, 5, or 6 bits**. Predict the number of levels before running.
Compare both a gradient and a photograph. The same number of levels can have different
visible effects depending on image content. For the photograph, levels are reduced per channel.


In [ ]:
my_bits = None  # FILL: choose 3, 5, or 6
if ready("F2", my_bits=my_bits):
    if my_bits in [3, 5, 6]:
        panels([gradient, quantize(gradient, my_bits)], ["8-bit ramp", f"{my_bits}-bit ramp"])
        panels([photo, quantize(photo, my_bits)], ["Reference photo", f"{my_bits} bits per channel"])
        print("Available levels per channel:", 2**my_bits)
    else:
        completion["F2"] = False
        print("Choose 3, 5, or 6.")


### QF2 · WRITE

**My chosen bits and predicted number of levels:** [Write here]  
**Observation:** Where is the effect easiest to see in each image? [Write here]  
**Conclusion:** Why is a smooth gradient a useful test even when a photograph looks acceptable? [Write here]


## G · Your investigation: choose the least costly adequate setting

You now have more freedom, but no new programming is required.
Choose **one task**, **one factor**, and **three different settings**.

| Task | Image | What counts as useful evidence? |
|---|---|---|
| Read PART A7 | `target` | Can you distinguish each character? |
| Separate the two bright markers | `target` | Do you see two separate spots? |
| Detect the thin crack | `target` | Is the thin zigzag visible in the grey rectangle? |
| Preserve a photo detail you name | `photo` | Is that named detail still distinguishable? |

| Factor | Use exactly this word | Settings you may choose |
|---|---|---|
| Width (pixels) | `"resolution"` | 24, 48, 96, 192 |
| Blur radius (pixels) | `"blur"` | 0, 1, 2, 4 |
| Bits per grayscale value / colour channel | `"bits"` | 2, 3, 4, 6, 8 |

**A controlled experiment:** change only your selected factor; keep image content and all
other settings unchanged. The tool displays the reference plus all three trials.
This is a human visual inspection, not a measured machine-learning accuracy benchmark.

**Write your plan before running:**

- **Task and image:** [Write here]
- **Factor and three settings:** [Write here]
- **What stays fixed:** [Write here]
- **Prediction:** [Write here]
- **Decision rule** (for example, “I can separate both markers”): [Write here]


### G1 · EXPERIMENT — Fill four short lines

Keep quotation marks around words. For example, a factor is written `"blur"`, not `blur`.
Choose your own settings from the table. Do not change the supplied comparison code below them.


In [ ]:
image_choice = ""                 # FILL: "target" or "photo"
factor = ""                       # FILL: "resolution", "blur", or "bits"
settings = [None, None, None]      # FILL: three DIFFERENT numbers from your factor's row
feature_to_inspect = ""           # FILL: short description of your task/detail

if ready("G1", image_choice=image_choice, factor=factor,
         first=settings[0], second=settings[1], third=settings[2],
         feature_to_inspect=feature_to_inspect):
    allowed = {"resolution": [24, 48, 96, 192], "blur": [0, 1, 2, 4], "bits": [2, 3, 4, 6, 8]}
    valid = (image_choice in ["target", "photo"] and factor in allowed
             and len(set(settings)) == 3 and all(v in allowed.get(factor, []) for v in settings))
    if valid:
        investigation_source = {"target": target, "photo": photo}[image_choice]
        trials = [trial_image(investigation_source, factor, v) for v in settings]
        panels([investigation_source] + trials, ["Reference"] + [f"{factor}: {v}" for v in settings])
        print("Inspect:", feature_to_inspect)
    else:
        completion["G1"] = False
        print("Check the image, factor, and three different settings against the table.")


### QG1 · WRITE

| Setting | Task succeeds, fails, or uncertain? | Specific visual evidence |
|---|---|---|
| Trial 1: ... | ... | ... |
| Trial 2: ... | ... | ... |
| Trial 3: ... | ... | ... |

**Conclusion (3–5 sentences):**
For resolution or bits, choose the smallest tested value that meets your decision rule.
For blur, identify the largest tested blur you can tolerate. If none succeeds, say so;
if all succeed, explain why you have not yet found the limit.
Justify your decision, explain the CV concept, and state one limitation of your experiment.

**My conclusion:** [Write here]


### G2 · EXPERIMENT — Does your conclusion transfer?

Keep your factor and the three settings exactly the same. Change only the image:
use `"photo"` if G1 used `"target"`, or `"target"` if G1 used `"photo"`.
Define a comparable visible-detail task on this second image (for example, small text versus fine clothing detail).
This tests transfer of a qualitative conclusion; the two images are not identical measurement targets.


In [ ]:
second_image = ""  # FILL: the OTHER image, "target" or "photo"
if not completion.get("G1", False):
    completion["G2"] = False
    print("Complete G1 first, then run this comparison.")
elif ready("G2", second_image=second_image):
    if second_image in ["target", "photo"] and second_image != image_choice:
        second_source = {"target": target, "photo": photo}[second_image]
        second_trials = [trial_image(second_source, factor, v) for v in settings]
        panels([second_source] + second_trials, ["Second reference"] + [f"{factor}: {v}" for v in settings])
        print("Same factor and settings:", factor, settings)
    else:
        completion["G2"] = False
        print("Choose the other image.")


### QG2 · WRITE

**Second image and comparable detail task:** [Write here]

| Same setting | Task succeeds, fails, or uncertain? | Evidence on the second image |
|---|---|---|
| Trial 1: ... | ... | ... |
| Trial 2: ... | ... | ... |
| Trial 3: ... | ... | ... |

**Conclusion:** Does the same setting still seem adequate? Explain why a setting that works
for one image or task may not be sufficient for another. [Write here]


## Finish · Explain the new concepts in your own words

Answer briefly, using evidence from this notebook:

1. Why can an image with many pixels still be poor for defect detection? Refer to focus or noise.
2. Why is choosing the lowest noise measurement not enough when the task is to detect a thin crack?
3. Give one difference between **aliasing** and **quantisation banding**.
4. Complete: “For my investigation, I would choose ___ because ___; this conclusion is limited by ___.”

**1:** [Write here]  
**2:** [Write here]  
**3:** [Write here]  
**4:** [Write here]


### RUN · Check code-task completion

This checks whether the required editable cells were filled and run in the current runtime.
It **cannot read or assess your written explanations**. It also cannot determine whether
your visual decisions are correct. After reconnecting to a fresh runtime, rerun cells in order;
C1 and E2 histories reset, so repeat their requested comparisons.


In [ ]:
required = ["identity", "A2", "B3", "C1_two_windows", "D2", "E2_three_steps", "F2", "G1", "G2"]
missing = [name for name in required if not completion.get(name, False)]
if missing:
    print("Still to fill/run:", ", ".join(missing))
else:
    print("Required editable code tasks were run. Now review your explanations and tables.")
print("Manual check: replace every [Write here] and every table '...' with your own work.")
print("Keep outputs visible in your submitted .ipynb file.")


### Q-check · WRITE

Did the check list any unfinished code tasks? Complete those first. Then name the one result
that most changed your initial expectation, and explain what you learned from it.

**My reflection:** [Write here]


### Before you submit

- Your name and student ID are correct.
- All code cells have been run in order; there are no unresolved error outputs.
- All required fill-in values are complete.
- Every question, experiment table, prediction, and conclusion contains your own answer.
- You tested both median windows in C1, all three sampling steps in E2, and both images in G1–G2.
- Your saved notebook includes experiment outputs. Your tables record comparisons from earlier runs.

Download the completed notebook using Colab's **File → Download → Download .ipynb**.
Suggested filename: **`YOUR_STUDENT_ID_Lab2_CV.ipynb`**.
Follow the course page / instructor announcement for the deadline, destination, and any
additional submission requirements. This notebook does not change the course grading policy.

### A small glossary

| Term | Meaning in this lab |
|---|---|
| Blur | Nearby scene details mix, reducing sharpness |
| Sensor noise | Unwanted variation in recorded measurements |
| Denoising | Reducing unwanted variation, potentially at a cost to real detail |
| Median filter | Replacing a value with the middle value in a local neighbourhood |
| Spatial resolution | Pixel dimensions for the fixed scene being represented |
| Interpolation | Estimating values on a new grid from existing samples |
| Aliasing | Coarse sampling makes fine detail appear as a misleading pattern |
| Quantisation | Rounding measurements to a limited set of possible values |

### Sources and image credits

- Course connection: camera, sensor, and digital-image concepts from the Computer Vision lectures.
- The inspection target and gradient are original, deterministic teaching graphics created for this lab.
- Photograph: Grace Hopper, distributed in Matplotlib's sample data as `grace_hopper.jpg`.
  See [Matplotlib's image examples](https://matplotlib.org/stable/gallery/images_contours_and_fields/image_demo.html).
- Filtering background: [OpenCV: Smoothing Images](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html).
- Sampling and anti-aliasing background: [scikit-image: Rescale, resize, and downscale](https://scikit-image.org/docs/stable/auto_examples/transform/plot_rescale.html).

The supplied code uses simple illustrative operations; it does not require scikit-image or OpenCV.
